# 03 — QLoRA SFT
Instruction fine-tuning on top of the DAPT checkpoint. Run `02` first.

**Runtime:** T4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/legal-compliance-slm"

In [ ]:
!git clone https://github.com/Shankar-behera/legal-compliance-slm.git
%cd legal-compliance-slm
!pip install -r requirements.txt -q

In [ ]:
import os, shutil
os.makedirs("data/processed", exist_ok=True)
for fname in ["sft_pairs.jsonl"]:
    src = f"{DRIVE_ROOT}/data/processed/{fname}"
    if os.path.exists(src):
        shutil.copy(src, f"data/processed/{fname}")
        print("restored", fname)
    else:
        print("MISSING:", src, "— run notebook 01 first")

# sft_config.yaml's base_model_path already points at the Drive DAPT
# checkpoint from notebook 02 — verify it exists before training
dapt_final = f"{DRIVE_ROOT}/checkpoints/dapt/final"
print("DAPT checkpoint present:", os.path.exists(dapt_final))

In [ ]:
from google.colab import userdata
import os, wandb

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
wandb.login(key=userdata.get("WANDB_API_KEY"))

## Run QLoRA SFT

Reads `configs/sft_config.yaml` + `configs/lora_config.yaml` (r=16, alpha=32, targeting q/k/v/o_proj). Trains with TRL's `SFTTrainer` for 3 epochs over the instruction pairs, holding out 10% for eval.

In [ ]:
from src.training.train_sft import run_sft

adapter_dir = run_sft("configs/sft_config.yaml")
print("LoRA adapter saved to:", adapter_dir)

## Plot the loss curve (train + eval)

In [ ]:
from src.training.plot_curves import plot_curve
import glob

state_path = sorted(glob.glob(f"{adapter_dir}/../checkpoint-*/trainer_state.json"))[-1]
plot_curve(state_path, output_path="docs/training_curve_sft.png", title="SFT Loss (QLoRA, r=16)")

## Sanity-check a generation before moving to export

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

tokenizer = AutoTokenizer.from_pretrained(dapt_final)
base = AutoModelForCausalLM.from_pretrained(dapt_final, device_map="auto", torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, adapter_dir)
model.eval()

prompt = (
    "You are a legal compliance auditor. Review the following scenario, "
    "identify any clause violations, and recommend remediation.\n\n"
    "Scenario:\nA vendor retains customer PII for 7 years with no documented retention basis.\n\nAudit:"
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(output[0], skip_special_tokens=True)[len(prompt):].strip())